In [4]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("data/knowledge_base.csv")

df = pd.read_csv(DATA_PATH)
df = df.fillna("")

print(df.shape)
df.head()

(10, 7)


,id,category,title,content,coach_name,skill_level,location
0,policy-001,cancellation,Private lesson cancellation,Private lessons must be cancelled at least 24 ...,,,Main Club
1,policy-002,booking,Court booking duration,Court bookings are available in 60-minute time...,,,Main Club
2,dropin-001,dropin,Beginner drop-in,The beginner drop-in is suitable for new and r...,,beginner,Main Club
3,coach-001,coach,Coach Amy profile,Coach Amy specializes in beginner fundamentals...,Coach Amy,beginner,Main Club
4,coach-002,coach,Coach David profile,Coach David specializes in intermediate and ad...,Coach David,advanced,Main Club


In [5]:
assert df["id"].is_unique
assert df["id"].notna().all()
assert df["content"].str.strip().ne("").all()

In [6]:
from minsearch import Index

documents = df.to_dict(orient="records")

text_fields = [
    "category",
    "title",
    "content",
    "coach_name",
    "skill_level",
    "location",
]

keyword_fields = ["id"]

index = Index(
    text_fields=text_fields,
    keyword_fields=keyword_fields,
)

index.fit(documents)

In [7]:
def minsearch_search(
    query: str,
    boost: dict[str, float] | None = None,
    num_results: int = 10,
) -> list[dict]:
    if boost is None:
        boost = {}

    return index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=num_results,
    )

In [8]:
results = minsearch_search(
    "How early should I cancel a private lesson?"
)

[(item["id"], item["title"]) for item in results[:5]]

[('course-001', 'Private lesson'),
 ('course-002', 'Semi-private lesson'),
 ('policy-001', 'Private lesson cancellation'),
 ('coach-001', 'Coach Amy profile'),
 ('faq-002', 'What to bring')]

In [9]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")
model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing.")

client = OpenAI(api_key=api_key)

In [10]:
question_generation_template = """
You are creating retrieval evaluation questions for a badminton club
AI assistant.

Generate exactly 3 questions that a real customer or front-desk
employee might ask, where the document below is the best source for
answering the question.

Requirements:
- Write natural user questions.
- Each question must be answerable from this document.
- Do not mention the document ID.
- Do not copy the title word-for-word in every question.
- Include a mixture of direct and conversational wording.
- Do not add facts that are absent from the document.
- Return valid JSON only.

Return this format:

[
  {{"question": "Question one"}},
  {{"question": "Question two"}},
  {{"question": "Question three"}}
]

Document ID: {id}
Category: {category}
Title: {title}
Content: {content}
Coach name: {coach_name}
Skill level: {skill_level}
Location: {location}
""".strip()

In [11]:
import json


def clean_json_response(text: str) -> str:
    text = text.strip()

    if text.startswith("```json"):
        text = text[len("```json"):]

    if text.startswith("```"):
        text = text[len("```"):]

    if text.endswith("```"):
        text = text[:-len("```")]

    return text.strip()


def generate_questions(document: dict) -> list[dict]:
    prompt = question_generation_template.format(**document)

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    cleaned = clean_json_response(response.output_text)
    questions = json.loads(cleaned)

    if not isinstance(questions, list):
        raise ValueError("The model did not return a JSON list.")

    output = []

    for item in questions:
        question = item.get("question", "").strip()

        if question:
            output.append(
                {
                    "id": document["id"],
                    "question": question,
                }
            )

    return output

In [12]:
sample_document = documents[0]

generated = generate_questions(sample_document)
generated

[{'id': 'policy-001',
  'question': 'How much notice do I need to give if I want to cancel my private lesson?'},
 {'id': 'policy-001',
  'question': 'What happens if I cancel my lesson less than 24 hours in advance?'},
 {'id': 'policy-001',
  'question': 'Are there any refund policies for late cancellations of private lessons?'}]

In [13]:
from time import sleep

from tqdm.auto import tqdm

ground_truth_results = []
generation_errors = []

for document in tqdm(documents):
    try:
        questions = generate_questions(document)
        ground_truth_results.extend(questions)

        # Small delay to reduce rate-limit risk
        sleep(0.2)

    except Exception as exc:
        generation_errors.append(
            {
                "id": document["id"],
                "error": str(exc),
            }
        )

  0%|          | 0/10 [00:00<?, ?it/s]

In [14]:
len(ground_truth_results), len(generation_errors)

(30, 0)

In [15]:
generation_errors[:10]

[]

In [16]:
failed_ids = {item["id"] for item in generation_errors}

retry_documents = [
    document
    for document in documents
    if document["id"] in failed_ids
]

for document in tqdm(retry_documents):
    try:
        questions = generate_questions(document)
        ground_truth_results.extend(questions)
    except Exception as exc:
        print(document["id"], exc)

0it [00:00, ?it/s]

In [17]:
df_questions = pd.DataFrame(ground_truth_results)

df_questions.head()

,id,question
0,policy-001,What is the cancellation policy for private le...
1,policy-001,How much notice do I need to give to cancel a ...
2,policy-001,Will I get a refund if I cancel my lesson late?
3,policy-002,How long can I book a court for my practice se...
4,policy-002,What time should I arrive before my court book...


In [18]:
print(df_questions.shape)
print(df_questions["id"].nunique())
print(df_questions.isna().sum())

(30, 2)
10
id          0
question    0
dtype: int64


In [19]:
df_questions = (
    df_questions
    .drop_duplicates(subset=["id", "question"])
    .reset_index(drop=True)
)

In [21]:
GROUND_TRUTH_PATH = Path("data/ground-truth-retrieval.csv")

df_questions.to_csv(
    GROUND_TRUTH_PATH,
    index=False,
)

print(f"Saved {len(df_questions)} questions to {GROUND_TRUTH_PATH}")

Saved 30 questions to data/ground-truth-retrieval.csv


In [22]:
df_merged = df_questions.merge(
    df[["id", "category", "title", "content"]],
    on="id",
    how="left",
)

df_merged[
    df_merged["category"] == "cancellation"
][["question", "title", "content"]]

,question,title,content
0,What is the cancellation policy for private le...,Private lesson cancellation,Private lessons must be cancelled at least 24 ...
1,How much notice do I need to give to cancel a ...,Private lesson cancellation,Private lessons must be cancelled at least 24 ...
2,Will I get a refund if I cancel my lesson late?,Private lesson cancellation,Private lessons must be cancelled at least 24 ...


In [23]:
df_questions = pd.read_csv(
    "data/ground-truth-retrieval.csv"
)

ground_truth = df_questions.to_dict(orient="records")

ground_truth[:3]

[{'id': 'policy-001',
  'question': 'What is the cancellation policy for private lessons?'},
 {'id': 'policy-001',
  'question': 'How much notice do I need to give to cancel a private lesson?'},
 {'id': 'policy-001',
  'question': 'Will I get a refund if I cancel my lesson late?'}]

In [24]:
def hit_rate(relevance_total: list[list[bool]]) -> float:
    if not relevance_total:
        return 0.0

    hits = sum(any(relevance) for relevance in relevance_total)

    return hits / len(relevance_total)


def mrr(relevance_total: list[list[bool]]) -> float:
    if not relevance_total:
        return 0.0

    total_score = 0.0

    for relevance in relevance_total:
        for rank, is_relevant in enumerate(relevance, start=1):
            if is_relevant:
                total_score += 1 / rank
                break

    return total_score / len(relevance_total)

In [25]:
def evaluate(
    ground_truth: list[dict],
    search_function,
) -> dict[str, float]:
    relevance_total = []

    for item in tqdm(ground_truth):
        expected_id = item["id"]
        question = item["question"]

        results = search_function(question)

        relevance = [
            result["id"] == expected_id
            for result in results
        ]

        relevance_total.append(relevance)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [26]:
baseline_metrics = evaluate(
    ground_truth=ground_truth,
    search_function=lambda question: minsearch_search(
        query=question,
        boost={},
        num_results=10,
    ),
)

baseline_metrics

  0%|          | 0/30 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.925}

In [27]:
evaluation_results = [
    {
        "approach": "minsearch_no_boost",
        **baseline_metrics,
    }
]

In [28]:
manual_boost = {
    "title": 2.0,
    "content": 1.0,
    "coach_name": 1.5,
    "category": 1.2,
}

In [29]:
manual_metrics = evaluate(
    ground_truth=ground_truth,
    search_function=lambda question: minsearch_search(
        query=question,
        boost=manual_boost,
        num_results=10,
    ),
)

manual_metrics

  0%|          | 0/30 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.9083333333333333}

In [30]:
from sklearn.model_selection import train_test_split

unique_ids = df_questions["id"].unique()

validation_ids, test_ids = train_test_split(
    unique_ids,
    test_size=0.3,
    random_state=42,
)

df_validation = df_questions[
    df_questions["id"].isin(validation_ids)
].reset_index(drop=True)

df_test = df_questions[
    df_questions["id"].isin(test_ids)
].reset_index(drop=True)

gt_validation = df_validation.to_dict(orient="records")
gt_test = df_test.to_dict(orient="records")

print("Validation questions:", len(gt_validation))
print("Test questions:", len(gt_test))
print("Validation documents:", df_validation["id"].nunique())
print("Test documents:", df_test["id"].nunique())

Validation questions: 21
Test questions: 9
Validation documents: 7
Test documents: 3


In [31]:
import random


def simple_optimize(
    param_ranges: dict[str, tuple[float, float]],
    objective_function,
    n_iterations: int = 50,
    random_seed: int = 42,
) -> tuple[dict[str, float], float]:
    random.seed(random_seed)

    best_params = None
    best_score = float("-inf")

    for _ in tqdm(range(n_iterations)):
        current_params = {
            field: random.uniform(low, high)
            for field, (low, high) in param_ranges.items()
        }

        current_score = objective_function(current_params)

        if current_score > best_score:
            best_score = current_score
            best_params = current_params

    if best_params is None:
        raise RuntimeError("Optimization did not produce parameters.")

    return best_params, best_score

In [32]:
param_ranges = {
    "category": (0.0, 3.0),
    "title": (0.0, 4.0),
    "content": (0.0, 3.0),
    "coach_name": (0.0, 4.0),
    "skill_level": (0.0, 3.0),
    "location": (0.0, 2.0),
}

In [33]:
def objective(boost_params: dict[str, float]) -> float:
    metrics = evaluate(
        ground_truth=gt_validation,
        search_function=lambda question: minsearch_search(
            query=question,
            boost=boost_params,
            num_results=10,
        ),
    )

    return metrics["mrr"]

In [34]:
best_boost, best_validation_mrr = simple_optimize(
    param_ranges=param_ranges,
    objective_function=objective,
    n_iterations=50,
)

print("Best boost:")
print(best_boost)
print("Validation MRR:", best_validation_mrr)

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

...

  0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

  0%|          | 0/21 [00:00<?, ?it/s]

Best boost:
{'category': 2.6765387031145362, 'title': 0.3477553305176646, 'content': 1.2657654590558112, 'coach_name': 0.11918887775228137, 'skill_level': 0.6559139244108101, 'location': 1.0107105762067248}
Validation MRR: 0.9642857142857143


In [35]:
optimized_test_metrics = evaluate(
    ground_truth=gt_test,
    search_function=lambda question: minsearch_search(
        query=question,
        boost=best_boost,
        num_results=10,
    ),
)

optimized_test_metrics

  0%|          | 0/9 [00:00<?, ?it/s]

{'hit_rate': 1.0, 'mrr': 0.8333333333333334}

In [36]:
baseline_test_metrics = evaluate(
    ground_truth=gt_test,
    search_function=lambda question: minsearch_search(
        query=question,
        boost={},
        num_results=10,
    ),
)

manual_test_metrics = evaluate(
    ground_truth=gt_test,
    search_function=lambda question: minsearch_search(
        query=question,
        boost=manual_boost,
        num_results=10,
    ),
)

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

In [37]:
comparison = pd.DataFrame(
    [
        {
            "approach": "No boost",
            **baseline_test_metrics,
        },
        {
            "approach": "Manual boost",
            **manual_test_metrics,
        },
        {
            "approach": "Optimized boost",
            **optimized_test_metrics,
        },
    ]
)

comparison

,approach,hit_rate,mrr
0,No boost,1.0,0.888889
1,Manual boost,1.0,0.888889
2,Optimized boost,1.0,0.833333


In [38]:
def collect_failures(
    ground_truth: list[dict],
    search_function,
    top_k: int = 10,
) -> pd.DataFrame:
    failures = []

    for item in ground_truth:
        expected_id = item["id"]
        question = item["question"]

        results = search_function(question)
        retrieved_ids = [
            result["id"]
            for result in results[:top_k]
        ]

        if expected_id not in retrieved_ids:
            failures.append(
                {
                    "question": question,
                    "expected_id": expected_id,
                    "retrieved_ids": retrieved_ids,
                }
            )

    return pd.DataFrame(failures)

In [39]:
failure_df = collect_failures(
    ground_truth=gt_test,
    search_function=lambda question: minsearch_search(
        query=question,
        boost=best_boost,
        num_results=10,
    ),
)

print("Failures:", len(failure_df))
failure_df.head(20)

Failures: 0


""


In [42]:
if "expected_id" in failure_df.columns:
    failure_details = failure_df.merge(
        df[["id", "category", "title", "content"]],
        left_on="expected_id",
        right_on="id",
        how="left",
    )
else:
    failure_details = pd.DataFrame(
        columns=[
            "question",
            "expected_id",
            "title",
            "category",
            "retrieved_ids",
        ]
    )

failure_details[
    [
        "question",
        "expected_id",
        "title",
        "category",
        "retrieved_ids",
    ]
].head(20)

failure_details[
    [
        "question",
        "expected_id",
        "title",
        "category",
        "retrieved_ids",
    ]
].head(20)

,question,expected_id,title,category,retrieved_ids


In [43]:
def collect_rankings(
    ground_truth: list[dict],
    search_function,
) -> pd.DataFrame:
    rows = []

    for item in ground_truth:
        results = search_function(item["question"])
        result_ids = [result["id"] for result in results]

        try:
            rank = result_ids.index(item["id"]) + 1
        except ValueError:
            rank = None

        rows.append(
            {
                "question": item["question"],
                "expected_id": item["id"],
                "rank": rank,
            }
        )

    return pd.DataFrame(rows)

In [44]:
ranking_df = collect_rankings(
    ground_truth=gt_test,
    search_function=lambda question: minsearch_search(
        query=question,
        boost=best_boost,
        num_results=10,
    ),
)

ranking_df.sort_values(
    by="rank",
    ascending=False,
    na_position="first",
).head(20)

,question,expected_id,rank
4,Can I get coaching tailored to my skill level ...,course-001,2
7,Where do I go to rent a badminton racket?,faq-001,2
5,Where are the private lessons held?,course-001,2
0,How long can I book a court for my practice se...,policy-002,1
1,What time should I arrive before my court book...,policy-002,1
3,What is a private lesson in badminton?,course-001,1
2,Are there any specific time slots available fo...,policy-002,1
6,Can I rent a racket when I arrive at the club?,faq-001,1
8,Is there a chance that rackets might not be av...,faq-001,1


In [47]:
BEST_BOOST = {'category': 2.6765387031145362, 
              'title': 0.3477553305176646,
                'content': 1.2657654590558112, 
                'coach_name': 0.11918887775228137,
                  'skill_level': 0.6559139244108101,
                    'location': 1.0107105762067248}


In [48]:
def search(
    query: str,
    num_results: int = 5,
) -> list[dict]:
    return index.search(
        query=query,
        filter_dict={},
        boost_dict=BEST_BOOST,
        num_results=num_results,
    )

In [50]:
comparison.to_csv(
    "data/retrieval-evaluation-results.csv",
    index=False,
)

In [51]:
import json

with open(
    "data/best-minsearch-boost.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        best_boost,
        file,
        indent=2,
    )